# Exploratory Data Analysis — Customer Support Intent Dataset

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

sns.set_theme(style='whitegrid')

In [ ]:
train_df = pd.read_csv('../data/processed/train.csv')
val_df   = pd.read_csv('../data/processed/val.csv')
test_df  = pd.read_csv('../data/processed/test.csv')

df = pd.concat([train_df, val_df, test_df], ignore_index=True)
print(f'Total examples: {len(df):,}')
df.head()

## Class Distribution

In [ ]:
counts = df['label'].value_counts()
fig, ax = plt.subplots(figsize=(10, 5))
counts.plot(kind='bar', ax=ax, color=sns.color_palette('Set2', len(counts)))
ax.set_title('Class Distribution (All Splits)')
ax.set_xlabel('Intent')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2, p.get_height()), 
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('../results/eda_class_distribution.png', dpi=150)
plt.show()
print(counts)

## Text Length Analysis

In [ ]:
df['text_len'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

stats = df.groupby('label')[['text_len', 'word_count']].agg(['mean', 'median', 'std'])
print('Text length stats per category:')
print(stats.round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for label in df['label'].unique():
    subset = df[df['label'] == label]['text_len']
    axes[0].hist(subset, bins=30, alpha=0.5, label=label)
axes[0].set_title('Text Length Distribution by Intent')
axes[0].set_xlabel('Character Count')
axes[0].legend(fontsize=7)

df.boxplot(column='word_count', by='label', ax=axes[1], rot=30)
axes[1].set_title('Word Count by Intent')
axes[1].set_xlabel('Intent')
plt.suptitle('')
plt.tight_layout()
plt.savefig('../results/eda_text_length.png', dpi=150)
plt.show()

## Word Clouds per Intent

In [ ]:
intents = sorted(df['label'].unique())
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, intent in zip(axes.flatten(), intents):
    text = ' '.join(df[df['label'] == intent]['text'].tolist())
    wc = WordCloud(width=400, height=250, background_color='white', max_words=80).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(intent, fontsize=11, fontweight='bold')
    ax.axis('off')
plt.suptitle('Word Clouds per Intent Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/eda_wordclouds.png', dpi=150)
plt.show()

## Sample Examples per Category

In [ ]:
for intent in intents:
    print(f'\n--- {intent} ---')
    samples = df[df['label'] == intent]['text'].sample(5, random_state=42)
    for i, s in enumerate(samples, 1):
        print(f'  {i}. {s}')